# 3 Coding attention mechanisms

Self-attention is a mechanism that allows each position in the input sequence to
consider the relevancy of, or "attend to," all other positions in the same
sequence when computing the representation of the sequence.

In self-attention, the self refers to the mechanism's ability to compute
attention weights by relating differnt positions within a single input sequence.
It assesses and learns the relationship and dependencies between various parts
of the input itself, such as words in a sentence or pixels in an image.

## A simple self-attention mechanism without trainable weights

We begin with a simplified version of self-attention where the weights are
determined by the position of the token within the sequence. Later we will add
trainable weights.

Each word in the sequence is represented by a dimensional embedded vector
representing the specific token, x(1), x(2), etc.

In self-attention, the goal is to calculate context vectors z(1), z(2), etc.
for each element of the input sequence x(1), x(2), etc. The *context vector* can
be thought of as an enriched embedding vector.

To calculate z(2) from x(2), we create an embedding that contains information
about x(2) and all the other elements x(1) through x(T). This allows the LLM to
understand the relationship and relevance of words in a sentence in relationship
to each other.

Consider a simplified, small embedding for a sentence "Your journey starts with
one step"

In [1]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],  # step     (x^6)
    ]
)

We then calculate the intermediate attention scores between the query token
and each input token. This is done by computing the dot product of the query, x(2)
in this example, with every other input token.

In [ ]:
query = inputs[1]
attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(x_i, query)
print(attn_scores_2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


The dot product is the result of multiplying each element in a vector with the
same element in the second vector and then summing the products to yield a
scalar value. This scalar value quantifies how closely two vectors are aligned.
A higher dot product indicates a greater degree of alignment or similarity
between two vectors. In this case, it determines the extent to which each
element in the sequence focuses on, or "attends to" any other element in the
sequence.

The next step is to normalize each of the attention scores to obtain attention
weights that sum to 1. This helps maintain training stability in the LLM.

In [3]:
attn_weights_2_tmp = attn_scores_2 / attn_scores_2.sum()
print("Attention weights:", attn_weights_2_tmp)
print("Sum:", attn_weights_2_tmp.sum())

Attention weights: tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])
Sum: tensor(1.0000)


This is a simplified approach to normalization. In practice, it is more
common to use a *softmax* function for normalization which is better at
handling extreme values and produces a more favourable gradient in training.

In [4]:
def softmax_naive(x):
    return torch.exp(x) / torch.exp(x).sum(dim=0)

attn_weights_2_naive = softmax_naive(attn_scores_2)
print("Attention weights:", attn_weights_2_naive)
print("Sum:", attn_weights_2_naive.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


This naive version can fail with large or small input values. PyTorch has a proper implementation of *softmax*.

In [5]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
print("Attention weights:", attn_weights_2)
print("Sum:", attn_weights_2.sum())

Attention weights: tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])
Sum: tensor(1.)


Now that we've computed the normalized attention weights, the final step is
calculating the context vector z(2) by multiplying the embedded input tokens,
x(i), with the corresponding attention weights then summing the resulting vectors.

The context vector z(2) is the weighted sum of all input vectors obtained by
multiplying each input vector with its corresponding attention weight.

In [6]:
query = inputs[1]   # x(2)
context_vec_2 = torch.zeros(query.shape)
for i,x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

print(context_vec_2)


tensor([0.4419, 0.6515, 0.5683])


## Computing attention weights for all input tokens

Now that we've seen how to calculate the attention weights for one token in a 
sequence, we'll expand that to calculate teh attention weights for every token.